# ProofLens — Kaggle Execution Scaffold (`notebooks/kaggle_runner.ipynb`)

## Purpose
This notebook provides a thin, reproducible execution wrapper for running **ProofLens** on Kaggle Notebooks (where the ~8 GB `yangsangtai/tiny-genimage` dataset is mounted directly at `/kaggle/input/tiny-genimage`).

**Architectural Rule:** Business logic lives inside the `prooflens` Python package (`src/prooflens/`), never duplicated in notebook cells. This notebook only:
1. Detects whether it is running inside a Kaggle runtime or locally.
2. Installs the `prooflens` package.
3. Discovers or accepts the mounted dataset path.
4. Provides the **Phase 0B** entrypoint for full dataset auditing on Kaggle.
5. Provides the **Phase 1+** entrypoint for GPU training and evaluation.

In [ ]:
import os
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    DATASET_ROOT = Path(os.environ.get("PROOFLENS_DATASET_ROOT", "/kaggle/input/tiny-genimage"))
    WORKING_DIR = Path("/kaggle/working")
else:
    DATASET_ROOT = Path(os.environ.get("PROOFLENS_DATASET_ROOT", "../data/fixtures")).resolve()
    WORKING_DIR = Path("../artifacts/notebook_run").resolve()

print(f"Dataset Root: {DATASET_ROOT}")
print(f"Working Dir:  {WORKING_DIR}")

## Step 1: Install ProofLens Package
On Kaggle, attach the packaged repository (`prooflens_code.zip` generated via `python scripts/package.py`) or clone the GitHub repository, then install in editable mode.

In [ ]:
import prooflens

# On Kaggle after attaching or cloning the ProofLens repository, run:
# !pip install --no-deps -e .
print(f"ProofLens version: {prooflens.__version__}")

## Step 2: Phase 0B — Full Dataset Audit Entrypoint (Deferred to Phase 0B on Kaggle)
Run the `prooflens.data` scanner and audit summary on the mounted dataset without copying raw images.

In [ ]:
from prooflens.data import save_manifest_csv, scan_dataset_directory, summarize_manifest_audit

RUN_PHASE0B_AUDIT = False
if RUN_PHASE0B_AUDIT and DATASET_ROOT.exists():
    manifest_df = scan_dataset_directory(DATASET_ROOT, include_corrupted=True)
    save_manifest_csv(manifest_df, WORKING_DIR / "manifests" / "images.csv")
    audit_summary = summarize_manifest_audit(manifest_df)
    print(audit_summary)

## Step 3: Phase 1+ — GPU Training & Evaluation Entrypoint (Deferred to Phase 1)
Uses the exact same `prooflens.experiments.runner` module as local CPU smoke testing.

In [ ]:
from prooflens.experiments import execute_smoke_experiment, load_and_validate_experiment_config

RUN_PHASE1_TRAINING = False
if RUN_PHASE1_TRAINING and DATASET_ROOT.exists():
    config = load_and_validate_experiment_config(
        "../configs/experiments/kaggle_template.yaml",
        dataset_root_override=DATASET_ROOT,
        output_dir_override=WORKING_DIR / "outputs",
    )
    result = execute_smoke_experiment(config)
    print(result)